Install Required Libraries

In [2]:
!pip install sentence-transformers psycopg2-binary pandas tqdm

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Load Processed Dataset

In [1]:
import pandas as pd

df = pd.read_csv("processed_complaints.csv")

df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Timely response?,Complaint ID,label,text
0,2019-11-18,Credit card or prepaid card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,XXXX claimed they delivered a package to my ad...,NaN,DISCOVER BANK,MA,021XX,NaN,Consent provided,Web,2019-11-18,Yes,3442136,escalate_to_human,Product: Credit card or prepaid card Sub-produ...
1,2020-04-10,Credit card or prepaid card,General-purpose prepaid card,Trouble using the card,Trouble getting information about the card,I got a Brinks Money pre-paid card in the mail...,Company has responded to the consumer and the ...,Netspend Corporation,IL,60657,NaN,Consent provided,Web,2020-04-14,Yes,3601853,escalate_to_human,Product: Credit card or prepaid card Sub-produ...
2,2019-07-09,Credit card or prepaid card,Store credit card,Problem with a purchase shown on your statement,Card was charged for something you did not pur...,On XX/XX/XXXX I was called by a creditor Nelso...,NaN,Nelson Cruz & Associates LLC,TN,37043,NaN,Consent provided,Web,2019-07-09,Yes,3300820,escalate_to_human,Product: Credit card or prepaid card Sub-produ...
3,2020-07-10,Credit card or prepaid card,General-purpose credit card or charge card,Trouble using your card,Can't use card to make purchases,Around XX/XX/2020 i XXXX XXXX XXXX opened a cr...,NaN,CAPITAL ONE FINANCIAL CORPORATION,CA,937XX,NaN,Consent provided,Web,2020-07-10,Yes,3739698,escalate_to_human,Product: Credit card or prepaid card Sub-produ...
4,2019-06-24,Credit card or prepaid card,General-purpose credit card or charge card,"Advertising and marketing, including promotion...",Confusing or misleading advertising about the ...,Equifax sent me credit card suggestions to hel...,NaN,"EQUIFAX, INC.",OR,971XX,NaN,Consent provided,Web,2019-06-24,Yes,3285243,escalate_to_human,Product: Credit card or prepaid card Sub-produ...


Load Embedding Model

In [8]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Generate Embeddings for Complaint Text

In [3]:
texts = df["text"].tolist()

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True
)

Batches:   0%|          | 0/3417 [00:00<?, ?it/s]

Above took 3.5hours for converting embeddings

Save Embeddings to File

In [4]:
import numpy as np

np.save("complaint_embeddings.npy", embeddings)

Load Saved Embeddings

In [1]:
import numpy as np
embeddings = np.load("complaint_embeddings.npy")

Connect to PostgreSQL Database


In [3]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="postgres",
    user="postgres",
    password="Hybye321$"
)

cursor = conn.cursor()

Reload Processed Dataset


In [4]:
import pandas as pd

df = pd.read_csv("processed_complaints.csv")

In [5]:
print(len(df), len(embeddings))

218674 218674


Insert Data and Embeddings into PostgreSQL

In [6]:
from tqdm import tqdm

for i in tqdm(range(len(df))):

    cursor.execute(
        """
        INSERT INTO complaints
        (text, label, issue, sub_issue, embedding)
        VALUES (%s, %s, %s, %s, %s)
        """,
        (
            df.iloc[i]["text"],
            df.iloc[i]["label"],
            df.iloc[i]["Issue"],
            df.iloc[i]["Sub-issue"],
            embeddings[i].tolist()
        )
    )

conn.commit()

100%|██████████| 218674/218674 [11:17<00:00, 322.90it/s]


Test Vector Similarity Search (Top 3)

In [11]:
conn.rollback()

query = "My credit card was charged twice"

query_vector = model.encode(query)

vector_str = "[" + ",".join(map(str, query_vector)) + "]"

cursor.execute(
"""
SELECT text, label
FROM complaints
ORDER BY embedding <-> %s::vector
LIMIT 3
""",
(vector_str,)
)

results = cursor.fetchall()

results

[("Product: Credit card or prepaid card Sub-product: General-purpose credit card or charge card Issue: Problem with a purchase shown on your statement Sub-issue: Overcharged for something you did purchase with the card Complaint: I had made a couple of purchases using PayPal credit ( Synchrony Bank ). These purchases are showing up on my statement but they are showing up twice on my statements. They are showing the original valid charge, but they are showing another charge the very next day for the exact same amount which is a duplicate. This has happened for 4 transactions. On my statements, the 4 valid charges are showing and then a day later from each of the 4, another charge for the exact same amount. Synchrony insists I was not double charged and I have tried several times to get them to resolve this, they have not and keep insisting I wasn't double charged despite the double charges appearing on my statement for a few months and also showing on their website. I believe they have 